# Pokrycie kodu i testowanie mutacyjne — Mealie
**Zespół:** KacperS, Basia, KacperM &nbsp;|&nbsp; **Przedmiot:** Badanie jakości systemów &nbsp;|&nbsp; zajęcia 2, zadanie 2

## Zadanie
Dla testów czarnoskrzynkowych całego zespołu:

**A. Pokrycie** (`coverage.py`, instrukcje + gałęzie):
1. ile sumarycznie linii nie zostało pokrytych,
2. które testy wykonują poszczególne gałęzie,
3. które testy wykonują te same fragmenty kodu,
4. fragmenty kodu nieosiągalne dla danych zgodnych ze specyfikacją.

**B. Mutacje** — kolejno wprowadzamy niewielkie kontrolowane zmiany w badanych funkcjach. Po każdej zmianie:
uruchamiamy **cały** zestaw testów, sprawdzamy, czy testy wykryły zmianę, i wskazujemy test, który ją wykrył albo
powinien wykryć. Przed kolejną zmianą przywracamy oryginalny kod.

## Badane funkcje

| Osoba | Funkcja | Plik | Testy |
|---|---|---|---|
| KacperS | `create_recipe_slug` | `mealie/schema/recipe/recipe.py` | KS-01 … KS-10 |
| KacperS | `QueryFilterBuilder.combine_filters` | `mealie/services/query_filter/builder.py` | KS-11 … KS-19c |
| KacperS | `PlanRulesDay.from_date` | `mealie/schema/meal_plan/plan_rules.py` | KS-20 … KS-28 |
| Basia | `parse_amount` | `mealie/services/parser_services/brute/process.py` | B-01 … B-05 |
| Basia | `UnitConverter.can_convert` | `mealie/services/parser_services/parser_utils/unit_utils.py` | B-06 … B-10 |
| Basia | `UnitConverter.convert` | `mealie/services/parser_services/parser_utils/unit_utils.py` | B-11 … B-15 |
| KacperM | `clean_image` | `mealie/services/scraper/cleaner.py` | KM-25 … KM-36 |
| KacperM | `clean_ingredients` | `mealie/services/scraper/cleaner.py` | KM-01 … KM-10 |
| KacperM | `clean_duration` | `mealie/services/scraper/cleaner.py` | KM-11 … KM-24 |

## Narzędzia (w tym folderze)
- `runner_zadanie2.py` — uruchamia cały zestaw (83 przypadki) w osobnym procesie; z `--coverage` zapisuje pokrycie
  linii i gałęzi **osobno dla każdego przypadku** (konteksty `coverage.py`).
- `mutants_zadanie2.py` — lista 38 mutacji oraz funkcja, która wprowadza mutację do pliku źródłowego, uruchamia testy
  i **zawsze** przywraca oryginał (`try/finally` + kontrola, że plik jest identyczny jak przed zmianą).

Pokrycie i analiza dotyczą **tylko 9 badanych funkcji** (kod, który wywołują, nie jest analizowany).

## Przygotowanie

In [ ]:
import difflib
import re
import subprocess
import sys
import tempfile
from collections import defaultdict
from html import escape
from pathlib import Path

from IPython.display import HTML, Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "mealie").is_dir())
HERE = ROOT / "tests" / "black_box" / "kacper-s" / "zajecia2" / "zadanie2"
sys.path.insert(0, str(HERE))

import mutants_zadanie2 as mz

OUT = Path(tempfile.mkdtemp(prefix="zadanie2_"))

# Badane funkcje: nazwa -> (osoba, plik, nazwy w raporcie coverage, rozpoznanie testów po ścieżce/nazwie)
FUNCS = {
    "create_recipe_slug": ("KacperS", mz.RECIPE,     ["create_recipe_slug"],                    "test_create_recipe_slug.py::"),
    "combine_filters":    ("KacperS", mz.BUILDER,    ["QueryFilterBuilder.combine_filters"],    "test_combine_filters.py::"),
    "from_date":          ("KacperS", mz.PLAN_RULES, ["PlanRulesDay.from_date"],                "test_from_date.py::"),
    "parse_amount":       ("Basia",   mz.PROCESS,    ["parse_amount", "parse_amount.keep_looping"], "test_basia.py::test_parse_amount["),
    "can_convert":        ("Basia",   mz.UNITS,      ["UnitConverter.can_convert"],             "test_basia.py::test_can_convert["),
    "convert":            ("Basia",   mz.UNITS,      ["UnitConverter.convert"],                 "test_basia.py::test_convert"),
    "clean_image":        ("KacperM", mz.CLEANER,    ["clean_image"],                           "test_clean_image.py::"),
    "clean_ingredients":  ("KacperM", mz.CLEANER,    ["clean_ingredients"],                     "test_clean_ingredients.py::"),
    "clean_duration":     ("KacperM", mz.CLEANER,    ["clean_duration"],                        "test_clean_duration.py::"),
}

SINGLE = {"test_create_recipe_slug_exactly_max_length_is_not_truncated": "KS-08",
          "test_combine_filters_single_filter": "KS-11",
          "test_combine_filters_ignores_empty_parts": "KS-17",
          "test_combine_filters_keeps_or_filter_grouped": "KS-18",
          "test_from_date_full_week_gives_seven_distinct_days": "KS-26",
          "test_from_date_does_not_depend_on_system_locale": "KS-28"}


def case_id(node: str) -> str:
    m = re.search(r"\[((?:KS|B|KM)-\d+[a-z]?)", node)
    return m.group(1) if m else SINGLE.get(node.split("::")[-1], node.split("::")[-1])


def id_key(case: str):
    m = re.match(r"([A-Z]+)-(\d+)([a-z]?)", case)
    return (m.group(1), int(m.group(2)), m.group(3)) if m else (case, 0, "")


def func_of(node: str) -> str:
    return next(name for name, (*_, marker) in FUNCS.items() if marker in node)


def src(file: str, no: int) -> str:
    return (ROOT / file).read_text(encoding="utf-8").splitlines()[no - 1].strip()


def table(headers, rows) -> None:
    th = "".join(f"<th style='border:1px solid #8884;padding:4px 8px;text-align:left'>{escape(str(h))}</th>" for h in headers)
    body = "".join("<tr>" + "".join(
        f"<td style='border:1px solid #8884;padding:4px 8px;vertical-align:top'>{c}</td>" for c in r) + "</tr>" for r in rows)
    display(HTML(f"<table style='border-collapse:collapse;font-size:13px'><tr>{th}</tr>{body}</table>"))


def pct(a, b) -> str:
    return f"{100 * a / b:.0f}%" if b else "—"


def fmt_lines(lines) -> str:
    out, start, prev = [], None, None
    for n in sorted(lines):
        if start is None:
            start = prev = n
        elif n == prev + 1:
            prev = n
        else:
            out.append((start, prev)); start = prev = n
    if start is not None:
        out.append((start, prev))
    return ", ".join(str(a) if a == b else f"{a}–{b}" for a, b in out) or "—"


def ids(nodes) -> str:
    return ", ".join(sorted({case_id(n) for n in nodes}, key=id_key)) or "—"


GREEN, RED, GRAY = "#1a7f37", "#cf222e", "#6e7781"

In [ ]:
base = mz.run_suite(OUT / "baseline", coverage=True)
BASE = base["outcomes"]
FS = {k.replace("\\", "/"): v for k, v in base["cov"]["files"].items()}
PER_TEST = base["per_test"]

rows = []
for owner in ("KacperS", "Basia", "KacperM"):
    nodes = [n for n in BASE if FUNCS[func_of(n)][0] == owner]
    passed = sum(BASE[n]["outcome"] == "passed" for n in nodes)
    failed = [n for n in nodes if BASE[n]["outcome"] == "failed"]
    rows.append([owner, len(nodes), passed, len(failed), ids(failed)])
rows.append(["<b>razem</b>", len(BASE), sum(v["outcome"] == "passed" for v in BASE.values()),
             sum(v["outcome"] == "failed" for v in BASE.values()), ""])
table(["Osoba", "Przypadki", "PASS", "FAIL", "Przypadki FAIL (znane rozbieżności z zajęć 1)"], rows)

> Przypadki FAIL to rozbieżności między specyfikacją a implementacją wykryte na zajęciach 1. Nie przeszkadzają w pomiarze
> pokrycia (linie wykonują się niezależnie od wyniku asercji), a w części B są punktem odniesienia: mutację uznajemy za
> wykrytą, gdy jakikolwiek test **zmieni wynik** (PASS↔FAIL) albo test, który już był FAIL, **zawiedzie inaczej**
> (inny komunikat błędu, np. inna zwrócona wartość).

---
# Część A — pokrycie
## A1. Ile linii nie zostało pokrytych?

In [ ]:
SCOPE = {}      # funkcja -> zbiór instrukcji
BRANCHES = {}   # funkcja -> lista gałęzi (z, do)
rows, tot = [], defaultdict(int)
for name, (owner, file, qns, _) in FUNCS.items():
    fns = [FS[file]["functions"][q] for q in qns]
    executed = set().union(*(f["executed_lines"] for f in fns))
    missing = set().union(*(f["missing_lines"] for f in fns))
    br_exec = [tuple(b) for f in fns for b in f["executed_branches"]]
    br_miss = [tuple(b) for f in fns for b in f["missing_branches"]]
    SCOPE[name] = executed | missing
    BRANCHES[name] = sorted(br_exec + br_miss)
    n_st, n_br = len(executed | missing), len(br_exec) + len(br_miss)
    for k, v in (("st", n_st), ("cov", len(executed)), ("miss", len(missing)), ("br", n_br), ("brc", len(br_exec))):
        tot[k] += v
    miss_html = "<br>".join(f"{n}: <code>{escape(src(file, n))}</code>" for n in sorted(missing)) or "—"
    rows.append([owner, f"<code>{name}</code>", n_st, len(executed),
                 f"<b style='color:{RED if missing else GREEN}'>{len(missing)}</b>", pct(len(executed), n_st),
                 f"{len(br_exec)}/{n_br}", pct(len(br_exec), n_br), miss_html])
rows.append(["", "<b>razem</b>", tot["st"], tot["cov"], f"<b>{tot['miss']}</b>", f"<b>{pct(tot['cov'], tot['st'])}</b>",
             f"{tot['brc']}/{tot['br']}", f"<b>{pct(tot['brc'], tot['br'])}</b>", ""])
table(["Osoba", "Funkcja", "Instrukcje", "Pokryte", "Niepokryte", "Linie %", "Gałęzie", "Gałęzie %", "Niepokryte linie"], rows)
print(f"Sumarycznie niepokrytych linii: {tot['miss']} z {tot['st']} "
      f"({pct(tot['miss'], tot['st'])}); niepokrytych gałęzi: {tot['br'] - tot['brc']} z {tot['br']}")

## A2. Które testy wykonują poszczególne gałęzie?
Gałąź to przejście z linii z warunkiem (`if`, `case`, `for`, `while`) do jednej z możliwych następnych linii.
Ujemny numer docelowy oznacza wyjście z funkcji (np. `return` w warunku pętli `while`).
`clean_duration` wywołuje samą siebie dla `{"minValue": ...}` i list, dlatego KM-15 i KM-16 pojawiają się przy kilku
wzajemnie wykluczających się gałęziach (wykonują funkcję dwa razy).

In [ ]:
def arcs_of(node: str, file: str) -> set:
    return {tuple(a) for a in PER_TEST.get(node, {}).get(file, {}).get("arcs", [])}


def lines_of(node: str, file: str) -> set:
    return set(PER_TEST.get(node, {}).get(file, {}).get("lines", []))


def show_branches(name: str) -> None:
    owner, file, *_ = FUNCS[name]
    nodes = [n for n in BASE if func_of(n) == name]
    rows = []
    for a, b in BRANCHES[name]:
        hit = [n for n in nodes if (a, b) in arcs_of(n, file)]
        target = f"{b}: <code>{escape(src(file, b))}</code>" if b > 0 else "<i>wyjście z funkcji</i>"
        who = ids(hit) if hit else f"<b style='color:{RED}'>żaden — gałąź niepokryta</b>"
        rows.append([f"{a}: <code>{escape(src(file, a))}</code>", target, len(hit), who])
    display(Markdown(f"#### `{name}` ({owner})"))
    if not rows:
        print("Brak gałęzi: funkcja nie ma instrukcji warunkowych (coverage.py nie liczy try/except jako gałęzi). "
              f"Ścieżki wykonują: {ids(nodes)}")
        return
    table(["Warunek (linia)", "Przejście do", "Testy", "Które"], rows)


for name in FUNCS:
    show_branches(name)

## A3. Które testy wykonują te same fragmenty kodu?
Dla każdej funkcji grupuję przypadki o **identycznej ścieżce** — tym samym zbiorze wykonanych linii i gałęzi.
Przypadki z jednej grupy są z punktu widzenia pokrycia zamienne (ale mogą sprawdzać różne wartości wyniku).

In [ ]:
rows = []
summary = []
for name, (owner, file, *_) in FUNCS.items():
    groups = defaultdict(list)
    for n in BASE:
        if func_of(n) != name:
            continue
        lines = frozenset(lines_of(n, file) & SCOPE[name])
        arcs = frozenset(a for a in arcs_of(n, file) if a in set(BRANCHES[name]))
        groups[(lines, arcs)].append(n)
    for (lines, arcs), nodes in sorted(groups.items(), key=lambda g: -len(g[1])):
        if len(nodes) > 1:
            rows.append([owner, f"<code>{name}</code>", len(nodes), ids(nodes), fmt_lines(lines)])
    n_tests = sum(len(v) for v in groups.values())
    summary.append([owner, f"<code>{name}</code>", n_tests, len(groups)])

table(["Osoba", "Funkcja", "Przypadków", "Przypadki z tą samą ścieżką", "Wykonane linie"], rows)
display(Markdown("**Ile różnych ścieżek wykonania dają przypadki każdej funkcji:**"))
table(["Osoba", "Funkcja", "Przypadków", "Różnych ścieżek"], summary)

## A4. Fragmenty nieosiągalne dla danych zgodnych ze specyfikacją
Specyfikacja = sygnatura i docstring funkcji. Poniżej każdy niepokryty fragment z A1 oraz fragmenty pokryte **wyłącznie**
przez dane spoza specyfikacji, z oceną, czy da się do nich dojść poprawnymi danymi.

In [ ]:
def who_runs(name: str, line: int) -> str:
    file = FUNCS[name][1]
    return ids([n for n in BASE if line in lines_of(n, file)])

for name, line in [("clean_ingredients", 369), ("from_date", 33), ("clean_image", 151), ("clean_ingredients", 364)]:
    print(f"{name:<18} linia {line}: wykonują ją -> {who_runs(name, line)}")

| Funkcja | Fragment | Pokryty przez | Osiągalny dla danych ze specyfikacji? | Uzasadnienie |
|---|---|---|---|---|
| `clean_ingredients` | 368–369 `case [str()]:` → `return [...]` | tylko **KM-10** (krotka) | **Nie** | Każda lista jest przechwytywana wcześniej przez `case list(ingredients)`, więc ten `case` działa tylko dla innych sekwencji (np. krotki), a te są poza specyfikacją (`list \| str \| None`). Dla danych zgodnych ze specyfikacją to martwy kod — i właśnie przez niego KM-10 zwraca wynik zamiast `TypeError`. |
| `clean_ingredients` | 363→364 słownik jako składnik | — | **Nie** | Specyfikacja dopuszcza listę **napisów**; słownik w liście to nieudokumentowane zachowanie. |
| `clean_image` | 150–151 `case [{"@id": ...}]` | — | **Nie** | Docstring wymienia 4 obsługiwane struktury, `@id` nie ma wśród nich (nieudokumentowane wsparcie schema.org). |
| `from_date` | 32–33 `except KeyError: return unset` | tylko **KS-28** (polskie locale); linię 32 również KS-27a/b | **Nie** (przez dane) | Dla dowolnej `date` `strftime("%A")` w angielskim locale zawsze daje nazwę dnia z enuma. `unset` pojawia się tylko przy innym języku systemu — zależy od środowiska, nie od danych. |
| `clean_duration` | 495–496 `case bool(): return None` | — | **Tak** | `time_entry: Any`, a docstring mówi „anything else returns None" — `True`/`False` to poprawne dane. Brakuje przypadku. |
| `parse_amount` | 66–68, 70→71 napis zaczynający się od ułamka | — | **Tak** | np. `"½ cup sugar"` — typowy zapis składnika. Brakuje przypadku. |
| `parse_amount` | 76–77 liczba + ułamek unicode | — | **Tak** | np. `"1½ cups flour"`. Brakuje przypadku. |
| `parse_amount` | 82→83–84 jednostka zaczynająca się od `(` lub `-` | — | **Tak** | np. `"1-2 cups flour"` (zakres) albo `"2(500 ml) water"`. Brakuje przypadku. |

---
# Część B — mutacje
Każda komórka poniżej robi dla jednej zmiany:
1. pokazuje zmianę w kodzie (diff z numerem linii),
2. wprowadza ją do pliku źródłowego,
3. uruchamia **cały** zestaw 83 przypadków w nowym procesie,
4. przywraca oryginalny plik i sprawdza, że jest identyczny jak przed zmianą,
5. porównuje wyniki z przebiegiem bazowym i wskazuje testy, które zmianę wykryły.

Dla zmian niewykrytych podaję test, który **powinien** ją wykryć, i od razu sprawdzam to wejście na kodzie oryginalnym
i zmienionym. **Mutacja równoważna** to zmiana, której żadne dane nie odróżnią od oryginału — jej wykrycie jest niemożliwe.

In [ ]:
MUT = {m.id: m for m in mz.MUTANTS}
RESULTS = {}
assert mz.git_diff_is_clean(sorted({m.file for m in mz.MUTANTS})), "Pliki źródłowe mają niezatwierdzone zmiany — przerwij"

# Dla zmian niewykrytych: który test powinien je wykryć + wejście sprawdzające (import, wyrażenie)
SLUG = "from mealie.schema.recipe.recipe import create_recipe_slug"
PARSE = "from mealie.services.parser_services.brute.process import parse_amount"
UC = "from mealie.services.parser_services.parser_utils.unit_utils import UnitConverter"
IMG = "from mealie.services.scraper.cleaner import clean_image"
ING = "from mealie.services.scraper.cleaner import clean_ingredients"
DUR = "from mealie.services.scraper.cleaner import clean_duration"

SHOULD_DETECT = {
    "M01": ("równoważna", "Gdy długość == `max_length`, obcięcie `[:max_length]` nic nie zmienia, więc `>` i `>=` dają ten sam wynik. Najbliżej jest KS-08 (dokładnie 250 znaków) — i poprawnie nie widzi różnicy.", SLUG, "create_recipe_slug('b' * 250) == 'b' * 250"),
    "M03": ("brak przypadku", "KS-06 podaje `max_length=250` jawnie, a KS-08 nie przekracza limitu. Wykryłby ją przypadek z **domyślnym** `max_length` i nazwą dłuższą niż 250 znaków (wariant KS-06 bez `max_length`).", SLUG, "len(create_recipe_slug('a' * 300))"),
    "M14": ("brak przypadku", "Żaden przypadek nie kończy się cyfrą, więc pętla nigdy nie dochodzi do końca napisu. Wykryłby ją przypadek „sama liczba\", np. `\"2\"`.", PARSE, "parse_amount('2')"),
    "M15": ("równoważna", "`amount` jest zawsze nadpisywane w obu gałęziach `if end > 0` / `else`, więc wartość początkowa nigdy nie trafia do wyniku.", PARSE, "parse_amount('2 cups')"),
    "M16": ("brak przypadku", "Brak przypadku z przecinkiem dziesiętnym. Wykryłby ją np. `\"1,5 kg potatoes\"`.", PARSE, "parse_amount('1,5 kg potatoes')"),
    "M17": ("brak przypadku", "Brak przypadku z jednostką zaczynającą się od `(` lub `-` (fragment niepokryty w A1). Wykryłby ją np. zakres `\"1-2 cups flour\"`.", PARSE, "parse_amount('1-2 cups flour')"),
    "M24": ("równoważna", "`to_unit` jest wcześniej zamieniane na `pint.Unit`, a `qty.to(to_unit)` daje wynik dokładnie w tej jednostce — `converted.units` i `to_unit` są zawsze równe (sprawdzone także dla skrótu `\"g\"`).", UC, "UnitConverter().convert(1, 'kilogram', 'g')"),
    "M25": ("brak przypadku", "Wszystkie przypadki `convert` mają ilość ≥ 0. Wykryłby ją przypadek z ujemną ilością, np. `convert(-1, \"kilogram\", \"gram\")` → `-1000`.", UC, "UnitConverter().convert(-1, 'kilogram', 'gram')[0]"),
    "M27": ("brak przypadku", "Żadna lista w KM-27 nie zawiera pustych wartości. Wykryłby ją np. `[URL, \"\", None]` → `[URL]`.", IMG, "clean_image(['https://e.com/a.jpg', '', None])"),
    "M28": ("brak przypadku", "W KM-28 każdy słownik ma `url`. Wykryłby ją przypadek z mieszaną listą, np. `[{\"url\": A}, {\"src\": B}]` → `[A]`.", IMG, "clean_image([{'url': 'https://e.com/a.jpg'}, {'src': 'https://e.com/b.jpg'}])"),
    "M32": ("brak przypadku", "KM-02 ma jedną linię. Wykryłby ją napis z pustą linią, np. `\"1 cup flour\\n\\n2 eggs\"`.", ING, "clean_ingredients('1 cup flour\\n\\n2 eggs')"),
    "M35": ("równoważna", "Dla `seconds == 0` przechodzi warunek, ale `round(0) or None` i tak zwraca `None` — tak jak oryginał (KM-20, KM-22).", DUR, "[clean_duration(0), clean_duration('PT0S')]"),
    "M36": ("brak przypadku", "KM-24 testuje wartość daleko powyżej limitu. Wykryłby ją przypadek **dokładnie na granicy** `MAX_DURATION_SECONDS` (2³¹−1 s).", DUR, "clean_duration((2**31 - 1) / 60)"),
    "M37": ("brak przypadku", "Brak przypadku z wartością logiczną (fragment niepokryty w A1). Wykryłby ją `clean_duration(True)` → `None`.", DUR, "clean_duration(True)"),
}


# Skrót do tabeli zbiorczej: jaki przypadek powinien wykryć zmianę
SHORT = {
    "M01": "— (równoważna; najbliżej KS-08)",
    "M03": "wariant KS-06 z domyślnym max_length: nazwa 300 znaków → slug 250 znaków",
    "M14": "B-xx: sama liczba, np. \"2\" → (2.0, \"\", \"\")",
    "M15": "— (równoważna)",
    "M16": "B-xx: przecinek dziesiętny, np. \"1,5 kg potatoes\" → 1.5",
    "M17": "B-xx: zakres, np. \"1-2 cups flour\" → cały napis w note",
    "M24": "— (równoważna)",
    "M25": "B-xx: ujemna ilość, convert(-1, kilogram, gram) → -1000",
    "M27": "wariant KM-27 z pustymi wartościami: [URL, \"\", None] → [URL]",
    "M28": "wariant KM-28 ze słownikiem bez url → pominięty",
    "M32": "wariant KM-02 z pustą linią → pusta linia pominięta",
    "M35": "— (równoważna; najbliżej KM-20, KM-22)",
    "M36": "wariant KM-24 dokładnie na granicy 2³¹−1 s → 2147483647",
    "M37": "KM-xx: clean_duration(True) → None",
}


def run_probe(import_line: str, expr: str) -> str:
    """Oblicza wyrażenie w nowym procesie (aktualny kod z dysku) i zwraca wynik albo nazwę wyjątku."""
    code = (f"{import_line}\ntry:\n    r = repr({expr})\nexcept Exception as e:\n    r = f'{{type(e).__name__}}: {{e}}'\n"
            f"print('RESULT:', r)")
    env = {**mz.os.environ, "PYTHONPYCACHEPREFIX": tempfile.mkdtemp(prefix="pyc_"), "PRODUCTION": "True", "TESTING": "True",
           "PYTHONIOENCODING": "utf-8"}
    p = subprocess.run([sys.executable, "-c", code], cwd=ROOT, capture_output=True, text=True, encoding="utf-8", env=env)
    line = next((l for l in p.stdout.splitlines() if l.startswith("RESULT:")), "RESULT: " + p.stderr.strip()[-200:])
    return line[len("RESULT: "):][:160]


def show_diff(m: mz.Mutant) -> None:
    text = (ROOT / m.file).read_text(encoding="utf-8")
    first = text[: text.index(m.original)].count("\n") + 1
    old, new = m.original.splitlines(), m.mutated.splitlines()
    out = []
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, old, new).get_opcodes():
        if tag == "equal":
            continue
        out += [f"- {first + i:>4} | {old[i]}" for i in range(i1, i2)]
        out += [f"+ {first + i1:>4} | {new[j]}" for j in range(j1, j2)]
    display(Markdown(f"**{m.id}** · `{m.function}` ({m.owner}) · {m.description}  \n`{m.file}`\n```diff\n" + "\n".join(out) + "\n```"))


def mutate(mid: str) -> None:
    m = MUT[mid]
    show_diff(m)
    with mz.applied(m):                                   # 2. wprowadź zmianę …
        outcomes = mz.run_suite(OUT / mid)["outcomes"]    # 3. … uruchom cały zestaw …
    restored = mz.git_diff_is_clean([m.file])             # 4. … przywróć i sprawdź
    changed = mz.compare(BASE, outcomes)                  # 5. porównaj z bazą
    RESULTS[mid] = changed
    print(f"Testy: {sum(v['outcome'] == 'passed' for v in outcomes.values())} PASS / "
          f"{sum(v['outcome'] == 'failed' for v in outcomes.values())} FAIL | oryginał przywrócony: {'tak' if restored else 'NIE!'}")
    if changed:
        display(HTML(f"<b style='color:{GREEN}'>WYKRYTA</b> przez {len(changed)} przypadk(i/ów):"))
        table(["Przypadek", "Zmiana wyniku", "Komunikat po zmianie"],
              [[case_id(c["node"]), c["kind"], f"<code>{escape(c['after'].splitlines()[0][:140] if c['after'] else '—')}</code>"]
               for c in sorted(changed, key=lambda c: id_key(case_id(c["node"])))])
        return
    kind, why, imp, expr = SHOULD_DETECT[mid]
    color = GRAY if kind == "równoważna" else RED
    display(HTML(f"<b style='color:{color}'>NIEWYKRYTA</b> ({kind}). {escape(why)}"))
    before = run_probe(imp, expr)
    with mz.applied(m):
        after = run_probe(imp, expr)
    verdict = "takie same → zmiany nie da się wykryć" if before == after else "różne → ten przypadek wykryłby zmianę"
    table(["Wejście sprawdzające", "Oryginał", "Po zmianie", "Wynik"],
          [[f"<code>{escape(expr)}</code>", f"<code>{escape(before)}</code>", f"<code>{escape(after)}</code>", verdict]])

## `create_recipe_slug` (KacperS)

In [ ]:
mutate("M01")

In [ ]:
mutate("M02")

In [ ]:
mutate("M03")

In [ ]:
mutate("M04")

In [ ]:
mutate("M05")

## `combine_filters` (KacperS)

In [ ]:
mutate("M06")

In [ ]:
mutate("M07")

In [ ]:
mutate("M08")

In [ ]:
mutate("M09")

In [ ]:
mutate("M10")

## `from_date` (KacperS)

In [ ]:
mutate("M11")

In [ ]:
mutate("M12")

In [ ]:
mutate("M13")

## `parse_amount` (Basia)

In [ ]:
mutate("M14")

In [ ]:
mutate("M15")

In [ ]:
mutate("M16")

In [ ]:
mutate("M17")

In [ ]:
mutate("M18")

## `can_convert` (Basia)

In [ ]:
mutate("M19")

In [ ]:
mutate("M20")

In [ ]:
mutate("M21")

## `convert` (Basia)

In [ ]:
mutate("M22")

In [ ]:
mutate("M23")

In [ ]:
mutate("M24")

In [ ]:
mutate("M25")

## `clean_image` (KacperM)

In [ ]:
mutate("M26")

In [ ]:
mutate("M27")

In [ ]:
mutate("M28")

In [ ]:
mutate("M29")

## `clean_ingredients` (KacperM)

In [ ]:
mutate("M30")

In [ ]:
mutate("M31")

In [ ]:
mutate("M32")

In [ ]:
mutate("M33")

## `clean_duration` (KacperM)

In [ ]:
mutate("M34")

In [ ]:
mutate("M35")

In [ ]:
mutate("M36")

In [ ]:
mutate("M37")

In [ ]:
mutate("M38")

---
## Podsumowanie mutacji

In [ ]:
rows = []
for mid, m in MUT.items():
    changed = RESULTS.get(mid)
    if changed is None:
        status = "nie uruchomiono"
    elif changed:
        status = f"<b style='color:{GREEN}'>wykryta</b>"
    elif SHOULD_DETECT[mid][0] == "równoważna":
        status = f"<b style='color:{GRAY}'>równoważna</b>"
    else:
        status = f"<b style='color:{RED}'>niewykryta</b>"
    kinds = sorted({c["kind"] for c in changed or []})
    who = ids([c["node"] for c in changed]) if changed else "—"
    should = escape(SHORT.get(mid, "")) if not changed else ""
    rows.append([mid, m.owner, f"<code>{m.function}</code>", escape(m.description), status, who, ", ".join(kinds), should])
table(["ID", "Osoba", "Funkcja", "Zmiana", "Wynik", "Wykryta przez", "Jak", "Powinien wykryć (brakujący przypadek)"], rows)

In [ ]:
rows = []
for owner in ("KacperS", "Basia", "KacperM", None):
    ms = [m for m in mz.MUTANTS if owner in (None, m.owner)]
    detected = sum(bool(RESULTS.get(m.id)) for m in ms)
    equivalent = sum(not RESULTS.get(m.id) and SHOULD_DETECT.get(m.id, ("",))[0] == "równoważna" for m in ms)
    survived = len(ms) - detected - equivalent
    rows.append([owner or "<b>razem</b>", len(ms), detected, survived, equivalent,
                 f"<b>{pct(detected, len(ms))}</b>", f"<b>{pct(detected, len(ms) - equivalent)}</b>"])
table(["Osoba", "Mutacje", "Wykryte", "Niewykryte", "Równoważne", "Wynik mutacyjny", "Bez równoważnych"], rows)

only_message = [mid for mid, ch in RESULTS.items() if ch and all(c["kind"] == "failed, inny komunikat" for c in ch)]
only_fixed = [mid for mid, ch in RESULTS.items() if ch and all(c["kind"] == "failed → passed" for c in ch)]
print("Wykryte wyłącznie przez zmianę komunikatu testu, który już był FAIL:", ", ".join(only_message) or "—")
print("Wykryte wyłącznie dlatego, że test FAIL zaczął przechodzić:", ", ".join(only_fixed) or "—")
print("Pliki źródłowe po wszystkich mutacjach identyczne z repozytorium:",
      "tak" if mz.git_diff_is_clean(sorted({m.file for m in mz.MUTANTS})) else "NIE!")

---
# Wnioski

## Część A: pokrycie

| Pytanie | Odpowiedź |
|---|---|
| **1. Ile linii nie pokryto?** | **10 z 123 instrukcji** (92% pokrycia linii) i **6 z 70 gałęzi** (91% pokrycia gałęzi). 7 z tych linii jest w `parse_amount` (Basia), po jednej w `clean_image`, `clean_ingredients` i `clean_duration` (KacperM). Funkcje KacperS oraz `can_convert` i `convert` (Basia) mają 100% pokrycia. |
| **2. Testy odpowiedzialne za gałęzie** | Tabele w A2. Wiele gałęzi wykonuje tylko jeden przypadek, np. obcięcie sluga (KS-06, KS-07), błąd 400 w `combine_filters` (KS-19b), wyjście z pętli w `parse_amount` (B-05), wiele `case` w `clean_image` i `clean_duration`. |
| **3. Testy wykonujące te same fragmenty** | Tabele w A3. Największe grupy to 8 przypadków `from_date` (KS-20…KS-26, jedna ścieżka), 6 przypadków `create_recipe_slug` (KS-01…05, 08) oraz po 5 przypadków `combine_filters` i `clean_image`. Najbardziej zróżnicowane są przypadki `parse_amount` (5 przypadków, 5 ścieżek) i `clean_duration` (14 przypadków, 10 ścieżek). |
| **4. Fragmenty nieosiągalne dla danych ze specyfikacji** | `clean_ingredients` 368–369 (`case [str()]`, przechwycony wcześniej przez `case list()`, działa tylko dla krotki z KM-10), `clean_ingredients` 364 (słownik w liście), `clean_image` 151 (`@id`) i `from_date` 33 (`unset`, osiągalny tylko przez zmianę języka systemu). Pozostałe niepokryte fragmenty (`parse_amount`: ułamki unicode, zakresy; `clean_duration`: `bool`) **są osiągalne** — brakuje dla nich przypadków. |

## Część B: mutacje

| | Mutacje | Wykryte | Niewykryte | Równoważne | Wynik | Bez równoważnych |
|---|---|---|---|---|---|---|
| KacperS | 13 | 11 | 1 | 1 | 85% | 92% |
| Basia | 12 | 6 | 4 | 2 | 50% | 60% |
| KacperM | 13 | 7 | 5 | 1 | 54% | 58% |
| **razem** | **38** | **24** | **10** | **4** | **63%** | **71%** |

1. **Pełne pokrycie nie gwarantuje wykrycia zmian.** `convert` (Basia) ma 100% pokrycia linii, a mimo to zmiana `quantity` → `abs(quantity)` (M25) przechodzi niezauważona, bo żaden przypadek nie ma ujemnej ilości. Tak samo `create_recipe_slug`: 100% pokrycia, a zmiana domyślnego `max_length` (M03) przechodzi, bo jedyny test długiej nazwy podaje `max_length` jawnie. Pokrycie mówi, *które linie się wykonały*, a mutacje — *czy testy sprawdzają, co te linie robią*.
2. **Niewykryte mutacje to głównie brakujące wartości graniczne i warianty danych:** dokładnie górna granica czasu (M36), sama liczba bez jednostki (M14), przecinek dziesiętny (M16), puste elementy listy (M27), słownik bez `url` (M28), puste linie (M32), `True` jako czas (M37). Każdą z nich wykrywa jedno proste wejście pokazane w części B. Dla każdej sprawdziliśmy to, uruchamiając je na oryginale i na mutancie.
3. **Część zmian jest niewykrywalna z zasady (mutacje równoważne).** M01, M15, M24 i M35 nie zmieniają działania funkcji dla żadnych danych. Nie świadczą o słabości testów, ale o nadmiarowym kodzie: martwa inicjalizacja `amount = 0.0` w `parse_amount`, a w `clean_duration` warunek `0 <` dublowany przez `round(seconds) or None`.
4. **Testy, które już wcześniej nie przechodziły, też wykrywają zmiany.** M05, M13, M18 i M29 wykryliśmy tylko dlatego, że test FAIL zawiódł *inaczej*. M02 (obcięcie o 1 znak) wykrył KS-07 — test, który *zaczął przechodzić*, bo zmiana przypadkiem usunęła wiszący myślnik. Bez kryterium „zmiana komunikatu” wynik zespołu spadłby z 24 do 20 wykrytych mutacji. Wszystkie 5 przypadków `parse_amount` kończy się FAIL, więc jedyna wykryta mutacja tej funkcji (M18) to zasługa tego kryterium.
5. **Różnice między osobami wynikają z doboru danych, a nie z liczby przypadków.** KacperM ma najwięcej przypadków (36), ale część z nich wykonuje tę samą ścieżkę (A3) i omija granice (`MAX_DURATION_SECONDS`, `bool`, puste elementy). KacperS ma podobną liczbę (32) i najwyższy wynik, bo przypadki brzegowe celują w konkretne warunki (pusta część filtra, filtr z `OR`, błąd 400, zły typ daty).
6. **Rekomendacje:** dodać ok. 10 przypadków z kolumny „Powinien wykryć” w tabeli zbiorczej. Po ich dodaniu wynik mutacyjny bez mutacji równoważnych wyniósłby 100% dla tego zestawu mutacji. Do tego warto usunąć martwy kod wskazany w A4 (`case [str()]` w `clean_ingredients`) i M15 (`amount = 0.0`).

## Jak odtworzyć
*Restart & Run All* w tym notebooku (kernel z `.venv`, który ma `coverage` i `pytest`; ok. 4–5 minut). Każda mutacja
jest wprowadzana do pliku źródłowego tylko na czas jednego uruchomienia testów i zawsze przywracana. Ostatnia komórka
sprawdza, czy pliki są identyczne z repozytorium.